This notebook extends the ["AI Agent (Part 2)"](./AI%20Agent%20(Part%202).ipynb) example.

In [ ]:
!pip install -q langchain langchain-openai

In [ ]:
from IPython.display import Image
from google.colab import userdata
from langchain.agents import AgentState
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.runnables import Runnable, RunnableConfig
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.base import BaseCheckpointSaver
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.prebuilt import ToolNode
from pathlib import Path
from pydantic import SecretStr
from typing import List

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

def display_graph(runnable: Runnable, output_png: Path):
    with output_png.open(mode="wb") as file:
        file.write(runnable.get_graph().draw_mermaid_png())

    display(Image(output_png, format="png"))

def explore_checkpoints(checkpoint_saver: BaseCheckpointSaver, config: RunnableConfig):
    checkpoints = list(checkpoint_saver.list(config))
    print(f"There are {len(checkpoints)} checkpoints in total:")
    for checkpoint in reversed(checkpoints):
        print(checkpoint)

def explore_state_history(compiled_state_graph: CompiledStateGraph, config: RunnableConfig):
    state_history = list(compiled_state_graph.get_state_history(config))

    for snapshot in reversed(state_history):
        print(f"Step: {snapshot.metadata['step']}")
        print("Current state:")
        print(snapshot.values)
        print(f"Next: {snapshot.next}")
        print()

In [ ]:
@tool
def weather(city: str) -> str:
    """Return a (fake) current-weather report for a city."""
    data = {
        "sofia": "Sofia: 18 C, partly cloudy",
        "london": "London: 11 C, rainy",
        "tokyo": "Tokyo: 22 C, sunny",
    }
    return data.get(city.lower(), f"No data for {city}.")

@tool
def search(query: str) -> str:
    """Look up a term in the built-in mini-encyclopedia."""
    data = {
        "langgraph": "LangGraph is a library for building stateful, cyclic LLM apps.",
        "react": "ReAct is a prompting pattern: Reason then Act, in a loop.",
        "dag": "A DAG is a directed acyclic graph — no cycles allowed.",
    }
    return data.get(query.lower(), "(nothing found)")


SYSTEM_PROMPT = "You are a concise assistant. Use tools when useful."
TOOLS = [weather, search]

In [ ]:
model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key).bind_tools(TOOLS)
checkpointer = InMemorySaver()

class CustomAgentState(AgentState):
    model_calls: int

def on_start(state: CustomAgentState):
    print("Our graph execution started!")

def on_end(state: CustomAgentState):
    print("Our graph execution just ended!")

def before_model_node(state: CustomAgentState):
    model_call_id = state.get('model_calls', 0) + 1
    print('=' * 20)
    print(f"Starting model call #{model_call_id}...")
    return { "model_calls": model_call_id }

def after_model_node(state: CustomAgentState):
    model_call_id = state.get('model_calls', 0)
    print(f"Finished model call #{model_call_id}.")
    print('=' * 20)

def model_node(state: CustomAgentState):
    response = model.invoke([SystemMessage(SYSTEM_PROMPT), *state["messages"]])
    return { "messages": [response] }

In [ ]:
def has_pending_tool_calls(state: CustomAgentState) -> bool:
    messages = state.get("messages", [])
    if not messages:
        return False

    last_message = messages[-1]
    return isinstance(last_message, AIMessage) and last_message.tool_calls

In [ ]:
graph_builder = StateGraph(CustomAgentState)
graph_builder.add_node("on_start", on_start)
graph_builder.add_node("on_end", on_end)
graph_builder.add_node("before_model", before_model_node)
graph_builder.add_node("after_model", after_model_node)
graph_builder.add_node("model", model_node)
graph_builder.add_node("tools", ToolNode(TOOLS))

graph_builder.add_edge(START, "on_start")
graph_builder.add_edge("on_start", "before_model")
graph_builder.add_edge("before_model", "model")
graph_builder.add_edge("model", "after_model")
graph_builder.add_conditional_edges("after_model", lambda x: "tools" if has_pending_tool_calls(x) else "on_end", ["tools", "on_end"])
graph_builder.add_edge("tools", "before_model")
graph_builder.add_edge("on_end", END)
graph = graph_builder.compile(checkpointer=checkpointer)

In [ ]:
display_graph(graph, Path("/content/graph.png"))

In [ ]:
thread1_config = { "configurable": { "thread_id": "thread_17" } }
thread1_result = graph.invoke(
    input={
        "messages": [HumanMessage("What's the weather in Tokyo and what is LangGraph, briefly?")]
    },
    config=thread1_config
)

In [ ]:
print_conversation(thread1_result["messages"])

In [ ]:
explore_checkpoints(checkpointer, thread1_config)

In [ ]:
explore_state_history(graph, thread1_config)

In [ ]:
thread1_result = graph.invoke(
    input={
        "messages": [HumanMessage("And what about London?")]
    },
    config=thread1_config
)

In [ ]:
print_conversation(thread1_result["messages"])

In [ ]:
explore_checkpoints(checkpointer, thread1_config)

In [ ]:
explore_state_history(graph, thread1_config)